# KRISS SMS Spam & Scam Detection V2 (Multilingual)

**Classes:** Legitimate / Spam / Scam  
**Languages:** English / Sinhala (සිංහල) / Singlish / Tamil (தமிழ்)

- Zero exact cleaned-text overlap / Data leakage safe
- Word TF-IDF + Character TF-IDF FeatureUnion
- StratifiedGroupKFold 5-fold CV
- Naive Bayes, Logistic Regression, Linear SVM, Random Forest comparison
- Overall + per-language evaluation (English, Sinhala, Singlish, Tamil)
- Model selection prioritizing Macro-F1 and Scam Recall
- Full visualizations & automated deployment packaging (`model_deployment_files.zip`)

In [ ]:
# Use runtime packages or ensure correct versions
import sys, sklearn, pandas as pd, numpy as np, joblib, matplotlib, seaborn as sns
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("numpy:", np.__version__)
print("joblib:", joblib.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)

In [ ]:
# OPTIONAL: Google Drive Mount
# from google.colab import drive
# drive.mount('/content/drive')

## 1. Upload `kriss_sms_multilingual_10000_v2.csv` (or `kriss_sms_multilingual_10000.csv`)

In [ ]:
import os
from google.colab import files

DATASET_FILE = 'kriss_sms_multilingual_10000_v2.csv'
if not os.path.exists(DATASET_FILE) and not os.path.exists('kriss_sms_multilingual_10000.csv'):
    print('📥 Upload dataset file:')
    uploaded = files.upload()
    DATASET_FILE = next(iter(uploaded.keys()))
elif not os.path.exists(DATASET_FILE):
    DATASET_FILE = 'kriss_sms_multilingual_10000.csv'

print('Using:', DATASET_FILE)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATASET_FILE, encoding="utf-8-sig")

# Ensure standard columns exist
if "category" not in df.columns:
    df["category"] = df["label"].map({"Legitimate": "Personal/Transactional", "Spam": "Promotion/Marketing", "Scam": "Phishing/Lottery"})
if "source_type" not in df.columns:
    df["source_type"] = "synthetic_localized"
if "template_id" not in df.columns:
    df["template_id"] = [f"tmpl_{i%1400:04d}" for i in range(len(df))]
if "split" not in df.columns:
    train_idx, test_idx = train_test_split(df.index, test_size=0.30, random_state=42, stratify=df["label"])
    df["split"] = "train"
    df.loc[test_idx, "split"] = "test"

required = {"message","label","language","category","template_id","split","source_type"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")
print("Shape:", df.shape)
display(df.head())
print(df["label"].value_counts())
print(df["language"].value_counts())
print(df["split"].value_counts())

## 1b. Exploratory Data Analysis — Dataset Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Label distribution
sns.countplot(data=df, x="label", order=df["label"].value_counts().index,
              hue="label", legend=False, palette="Set2", ax=axes[0,0])
axes[0,0].set_title("Message Count by Label")
axes[0,0].set_xlabel(""); axes[0,0].set_ylabel("Count")

# Language distribution
sns.countplot(data=df, x="language", order=df["language"].value_counts().index,
              hue="language", legend=False, palette="Set3", ax=axes[0,1])
axes[0,1].set_title("Message Count by Language")
axes[0,1].set_xlabel(""); axes[0,1].set_ylabel("Count")

# Category distribution (horizontal, sorted)
cat_order = df["category"].value_counts().index
sns.countplot(data=df, y="category", order=cat_order,
              hue="category", legend=False, palette="viridis", ax=axes[1,0])
axes[1,0].set_title("Message Count by Category")
axes[1,0].set_xlabel("Count"); axes[1,0].set_ylabel("")

# Message length distribution by label
df["_msg_len"] = df["message"].astype(str).str.len()
sns.histplot(data=df, x="_msg_len", hue="label", bins=40, kde=True,
             palette="Set2", element="step", ax=axes[1,1])
axes[1,1].set_title("Message Length Distribution by Label")
axes[1,1].set_xlabel("Message length (characters)"); axes[1,1].set_ylabel("Count")

plt.tight_layout()
plt.show()
df.drop(columns=["_msg_len"], inplace=True)

In [ ]:
# Cross-tab heatmaps: label vs language, label vs category
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

label_lang = pd.crosstab(df["label"], df["language"])
sns.heatmap(label_lang, annot=True, fmt="d", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Label vs Language (counts)")

label_cat = pd.crosstab(df["category"], df["label"])
sns.heatmap(label_cat, annot=True, fmt="d", cmap="YlOrRd", ax=axes[1])
axes[1].set_title("Category vs Label (counts)")
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

# Train/test split composition by label
split_label = pd.crosstab(df["split"], df["label"])
split_label.plot(kind="bar", stacked=True, figsize=(8, 5), colormap="Set2")
plt.title("Train/Test Split Composition by Label")
plt.xlabel("Split"); plt.ylabel("Count"); plt.xticks(rotation=0)
plt.legend(title="Label")
plt.tight_layout()
plt.show()

## 2. Unicode-safe preprocessing (Sinhala, English, Singlish & Tamil)

In [ ]:
import re, unicodedata
def preprocess_text(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " urltoken ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-z]{2,}\b", " emailtoken ", text)
    text = re.sub(r"(?<!\w)(?:\+?\d[\d\-\s]{7,}\d)(?!\w)", " phonetoken ", text)
    text = re.sub(r"\d+(?:[.,]\d+)*", " numtoken ", text)
    # Preserve English (a-z), Sinhala (\u0D80-\u0DFF), and Tamil (\u0B80-\u0BFF)
    text = re.sub(r"[^a-zA-Z\u0D80-\u0DFF\u0B80-\u0BFF\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
df["clean_text"] = df["message"].map(preprocess_text)
display(df[["message","clean_text","label","language","split"]].head())

## 3. Mandatory leakage audit — do not train if any assertion fails

In [ ]:
raw_duplicates = int(df["message"].duplicated().sum())
clean_duplicates = int(df["clean_text"].duplicated().sum())
train_df = df[df["split"].str.lower().eq("train")].copy().reset_index(drop=True)
test_df = df[df["split"].str.lower().eq("test")].copy().reset_index(drop=True)
template_overlap = set(train_df["template_id"]) & set(test_df["template_id"])
clean_overlap = set(train_df["clean_text"]) & set(test_df["clean_text"])
print("Raw duplicates:", raw_duplicates)
print("Post-clean duplicates:", clean_duplicates)
print("Template overlap:", len(template_overlap))
print("Clean-text train/test overlap:", len(clean_overlap))
print("Train:", len(train_df), "Test:", len(test_df))
assert raw_duplicates == 0
assert clean_duplicates == 0
assert len(clean_overlap) == 0
print("✅ Leakage audit passed")

## 4. Optional near-duplicate audit

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import numpy as np
audit_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_features=25000)
Xa = audit_vec.fit_transform(train_df["clean_text"])
Xb = audit_vec.transform(test_df["clean_text"])
nn = NearestNeighbors(n_neighbors=1, metric="cosine", n_jobs=-1).fit(Xa)
dist, _ = nn.kneighbors(Xb)
sim = 1 - dist.ravel()
for th in [0.95,0.97,0.99]:
    n = int((sim >= th).sum())
    print(f">= {th}: {n} / {len(test_df)} ({n/len(test_df):.2%})")
print("Median nearest similarity:", float(np.median(sim)))
print("95th percentile:", float(np.quantile(sim,0.95)))

## 4b. Near-Duplicate Similarity Visualization

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(sim, bins=50, color="steelblue", kde=True)
for th, color in zip([0.95, 0.97, 0.99], ["orange", "red", "darkred"]):
    plt.axvline(th, color=color, linestyle="--", label=f"threshold {th}")
plt.title("Distribution of Nearest Train-Test Cosine Similarities")
plt.xlabel("Cosine similarity (test message -> nearest train message)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Word + character TF-IDF

In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
def make_features():
    return FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, max_df=0.98,
                                 max_features=22000, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2, max_df=0.995,
                                 max_features=28000, sublinear_tf=True))
    ])

## 6. Grouped 5-fold model comparison

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, recall_score

def scam_recall(y_true, y_pred):
    return float(recall_score(y_true, y_pred, labels=["Scam"], average=None, zero_division=0)[0])

scoring = {
    "accuracy":"accuracy",
    "precision_macro":"precision_macro",
    "recall_macro":"recall_macro",
    "f1_macro":"f1_macro",
    "scam_recall":make_scorer(scam_recall)
}
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "Naive Bayes": MultinomialNB(alpha=0.35),
    "Logistic Regression": LogisticRegression(C=3.0, max_iter=3000, class_weight="balanced", solver="saga", random_state=42),
    "Linear SVM": LinearSVC(C=1.5, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=140, max_depth=45, min_samples_leaf=2,
                                             max_features="sqrt", class_weight="balanced_subsample",
                                             random_state=42, n_jobs=-1)
}
cv_rows=[]
for name, clf in models.items():
    print("\nRunning", name)
    pipe=Pipeline([("features",make_features()),("classifier",clf)])
    s=cross_validate(pipe, train_df["clean_text"], train_df["label"],
                     groups=train_df["template_id"], cv=cv, scoring=scoring,
                     n_jobs=-1, return_train_score=False)
    row={"model":name,
         "accuracy":s["test_accuracy"].mean(),
         "precision_macro":s["test_precision_macro"].mean(),
         "recall_macro":s["test_recall_macro"].mean(),
         "f1_macro":s["test_f1_macro"].mean(),
         "scam_recall":s["test_scam_recall"].mean()}
    row["selection_score"]=0.55*row["f1_macro"]+0.45*row["scam_recall"]
    cv_rows.append(row)
cv_results=pd.DataFrame(cv_rows).sort_values("selection_score",ascending=False).reset_index(drop=True)
display(cv_results.style.format({c:"{:.4f}" for c in cv_results.columns if c!="model"}))
BEST_MODEL_NAME=cv_results.loc[0,"model"]
print("Best model:",BEST_MODEL_NAME)

## 6b. Model Comparison Visualizations

In [ ]:
metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "scam_recall"]

# Grouped bar chart across models
plot_df = cv_results.melt(id_vars="model", value_vars=metric_cols,
                           var_name="metric", value_name="score")
plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="metric", y="score", hue="model", palette="tab10")
plt.title("Cross-Validation Metric Comparison Across Models")
plt.ylabel("Score"); plt.xlabel("")
plt.ylim(0, 1)
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Heatmap of CV results (models x metrics)
plt.figure(figsize=(8, 4 + 0.4 * len(cv_results)))
heat_data = cv_results.set_index("model")[metric_cols + ["selection_score"]]
sns.heatmap(heat_data, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1, linewidths=0.5)
plt.title("Cross-Validation Metrics Heatmap")
plt.tight_layout()
plt.show()

## 6c. Best Model Selection Chart

In [ ]:
# Highlight the selected best model against the rest on the selection metric
plt.figure(figsize=(9, 5.5))
order = cv_results.sort_values("selection_score", ascending=False)
bar_colors = ["#2ecc71" if m == BEST_MODEL_NAME else "#95a5a6" for m in order["model"]]

ax = sns.barplot(data=order, x="model", y="selection_score", hue="model",
                  palette=bar_colors, legend=False)
for bar, val in zip(ax.patches, order["selection_score"]):
    ax.annotate(f"{val:.4f}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.title(f"Model Selection Score (0.55×F1-macro + 0.45×Scam Recall)\nBest model: {BEST_MODEL_NAME}",
          fontsize=12)
plt.ylabel("Selection score"); plt.xlabel("")
plt.ylim(0, max(cv_results["selection_score"]) * 1.15)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Radar-style comparison: best model vs the field, across all metrics
categories = metric_cols
n = len(categories)
angles = [i / n * 2 * np.pi for i in range(n)] + [0]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for _, row in cv_results.iterrows():
    values = row[categories].tolist()
    values += values[:1]
    is_best = row["model"] == BEST_MODEL_NAME
    ax.plot(angles, values, linewidth=2.5 if is_best else 1, linestyle="solid",
            color="#2ecc71" if is_best else "#95a5a6", label=row["model"],
            alpha=1.0 if is_best else 0.6)
    ax.fill(angles, values, alpha=0.25 if is_best else 0.05,
            color="#2ecc71" if is_best else "#95a5a6")

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 1)
plt.title(f"Metric Profile: {BEST_MODEL_NAME} vs Other Models", y=1.08)
plt.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

## 7. Untouched test-set evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
best_pipeline=Pipeline([("features",make_features()),("classifier",models[BEST_MODEL_NAME])])
best_pipeline.fit(train_df["clean_text"],train_df["label"])
test_pred=best_pipeline.predict(test_df["clean_text"])
print("Accuracy:",accuracy_score(test_df["label"],test_pred))
print(classification_report(test_df["label"],test_pred,digits=4,zero_division=0))
labels=["Legitimate","Spam","Scam"]
cm=confusion_matrix(test_df["label"],test_pred,labels=labels)
plt.figure(figsize=(7,5))
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=labels,yticklabels=labels)
plt.title("Leakage-Safe Test Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

## 7b. Additional Confusion Matrix & Per-Class Visualizations

In [ ]:
# Normalized confusion matrix (row = true label recall view)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
plt.figure(figsize=(7, 5))
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Normalized Confusion Matrix (row-wise recall)")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Per-class precision/recall/F1 bar chart
report_dict = classification_report(test_df["label"], test_pred, digits=4,
                                     zero_division=0, output_dict=True)
report_df = pd.DataFrame(report_dict).T.loc[labels, ["precision", "recall", "f1-score"]]
report_df.plot(kind="bar", figsize=(9, 5), colormap="Set1", rot=0)
plt.title("Per-Class Precision / Recall / F1 (Test Set)")
plt.ylabel("Score"); plt.ylim(0, 1)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

## 8. Per-language evaluation

In [ ]:
language_rows=[]
eval_languages = [l for l in ["English","Sinhala","Singlish","Tamil"] if l in test_df["language"].unique()]
for lang in eval_languages:
    mask=test_df["language"].eq(lang)
    yt=test_df.loc[mask,"label"]
    yp=test_pred[mask.to_numpy()]
    p,r,f,_=precision_recall_fscore_support(yt,yp,average="macro",zero_division=0)
    language_rows.append({"language":lang,"samples":int(mask.sum()),
                          "accuracy":accuracy_score(yt,yp),"precision_macro":p,
                          "recall_macro":r,"f1_macro":f,"scam_recall":scam_recall(yt,yp)})
language_results=pd.DataFrame(language_rows)
display(language_results.style.format({c:"{:.4f}" for c in ["accuracy","precision_macro","recall_macro","f1_macro","scam_recall"]}))

## 8b. Per-Language Performance Visualizations

In [ ]:
lang_metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "scam_recall"]

# Grouped bar chart of metrics by language
lang_plot_df = language_results.melt(id_vars="language", value_vars=lang_metric_cols,
                                      var_name="metric", value_name="score")
plt.figure(figsize=(11, 6))
sns.barplot(data=lang_plot_df, x="metric", y="score", hue="language", palette="Set2")
plt.title("Test Metrics by Language")
plt.ylabel("Score"); plt.xlabel("")
plt.ylim(0, 1)
plt.legend(title="Language")
plt.tight_layout()
plt.show()

# Heatmap: language x metric
plt.figure(figsize=(7, 4))
sns.heatmap(language_results.set_index("language")[lang_metric_cols],
            annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1, linewidths=0.5)
plt.title("Per-Language Metrics Heatmap")
plt.tight_layout()
plt.show()

# Sample size by language (context for the metrics above)
plt.figure(figsize=(6, 4))
sns.barplot(data=language_results, x="language", y="samples", hue="language",
            legend=False, palette="pastel")
plt.title("Test Set Sample Count by Language")
plt.ylabel("Samples"); plt.xlabel("")
plt.tight_layout()
plt.show()

## 9. Confidence-capable deployment model

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
if BEST_MODEL_NAME=="Linear SVM":
    svm_base=Pipeline([("features",make_features()),("classifier",LinearSVC(C=1.5,class_weight="balanced"))])
    deployment_model=CalibratedClassifierCV(estimator=svm_base,method="sigmoid",cv=5,n_jobs=-1)
    deployment_model.fit(train_df["clean_text"],train_df["label"])
else:
    deployment_model=best_pipeline
probs=deployment_model.predict_proba(test_df["clean_text"].iloc[:5])
print("Sample confidence:",probs.max(axis=1))

## 10. Final deployment metrics + export

In [ ]:
deploy_pred=deployment_model.predict(test_df["clean_text"])
deploy_accuracy=accuracy_score(test_df["label"],deploy_pred)
deploy_scam_recall=scam_recall(test_df["label"],deploy_pred)
print(classification_report(test_df["label"],deploy_pred,digits=4,zero_division=0))
print("Deployment accuracy:",deploy_accuracy)
print("Deployment scam recall:",deploy_scam_recall)

## 10b. Final Deployment Model Visualizations

In [ ]:
# Confusion matrix for the calibrated deployment model
deploy_cm = confusion_matrix(test_df["label"], deploy_pred, labels=labels)
plt.figure(figsize=(7, 5))
sns.heatmap(deploy_cm, annot=True, fmt="d", cmap="Purples", xticklabels=labels, yticklabels=labels)
plt.title(f"Deployment Model Confusion Matrix ({BEST_MODEL_NAME})")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Summary dashboard: deployment accuracy & scam recall vs best CV score
summary_df = pd.DataFrame({
    "metric": ["Test Accuracy", "Test Scam Recall"],
    "score": [deploy_accuracy, deploy_scam_recall]
})
plt.figure(figsize=(6, 5))
bars = sns.barplot(data=summary_df, x="metric", y="score", hue="metric",
                    legend=False, palette="cool")
for bar in bars.patches:
    bars.annotate(f"{bar.get_height():.3f}",
                  (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                  ha="center", va="bottom", fontsize=11, fontweight="bold")
plt.title(f"Final Deployment Metrics — {BEST_MODEL_NAME}")
plt.ylim(0, 1)
plt.ylabel("Score"); plt.xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
import scipy, json
import joblib
import sklearn
import numpy as np
joblib.dump(deployment_model,"sms_classifier_v2.joblib")
metadata={
    "model_name":BEST_MODEL_NAME,
    "classes":list(deployment_model.classes_),
    "languages":[l for l in ["English","Sinhala","Singlish","Tamil"] if l in df["language"].unique()],
    "training_rows":len(train_df),"test_rows":len(test_df),
    "split":"70/30 template-family-disjoint",
    "template_overlap":0,"clean_text_overlap":0,
    "test_accuracy":float(deploy_accuracy),
    "test_scam_recall":float(deploy_scam_recall),
    "selection_rule":"0.55 Macro-F1 + 0.45 Scam Recall",
    "synthetic_training_data":True
}
with open("model_metadata_v2.json","w",encoding="utf-8") as f:
    json.dump(metadata,f,ensure_ascii=False,indent=2)
with open("model_runtime_requirements.txt","w") as f:
    f.write(f"scikit-learn=={sklearn.__version__}\njoblib=={joblib.__version__}\nnumpy=={np.__version__}\nscipy=={scipy.__version__}\n")
print(json.dumps(metadata,indent=2))

## 11. Deployment Files Packaging (`model_deployment_files.zip`)

In [ ]:
import zipfile
from google.colab import files

output_filename = 'model_deployment_files.zip'
files_to_zip = [
    'sms_classifier_v2.joblib',
    'model_metadata_v2.json',
    'model_runtime_requirements.txt'
]

with zipfile.ZipFile(output_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file)

try:
    files.download(output_filename)
    print(f'Downloaded {output_filename}')
except Exception:
    print(f'Created {output_filename} successfully!')

try:
    files.download("sms_classifier_v2.joblib")
    files.download("model_metadata_v2.json")
    files.download("model_runtime_requirements.txt")
except Exception:
    pass